# Fourier-mode decay of Newton-generated KdV waves

Generate periodic traveling-wave profiles with `KdVNewton`, inspect how their Fourier coefficients decay with mode number, and evolve one profile with the KdV time solver. These are two different views: spectral decay is coefficient size **across mode number**; the time plot shows how each mode's magnitude changes **during evolution**. For an exact traveling wave, modal magnitudes are constant in time while their phases change.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

root = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "solvers").is_dir()), None)
if root is None:
    raise RuntimeError("Run this notebook from within the project so its solvers package can be found.")
sys.path.insert(0, str(root))

from solvers.pde import KdV
from solvers.spectral import KdVNewton

## Generate the traveling-wave profiles

`KdVNewton.phi_hat` uses coefficients normalized by `N`, so the physical profile is `irfft(N * phi_hat)`. The real-FFT index is the nonnegative mode number, with wavenumber `2 pi mode / P`.

In [ ]:
P = 2 * np.pi
N = 128
dx = P / N
mean = 0.0
target_amplitudes = [0.1, 0.25, 0.4]

waves = []
profiles = []
for target_amp in target_amplitudes:
    wave = KdVNewton(P=P, target_amp=target_amp, mean=mean, dx=dx)
    wave.solve_w_continuation(n_steps=20)
    waves.append(wave)
    profiles.append(np.fft.irfft(N * wave.phi_hat, n=N))

modes = np.arange(N // 2 + 1)
wavenumbers = 2 * np.pi * modes / P
coefficient_magnitudes = np.array([np.abs(wave.phi_hat) for wave in waves])

print(f"Generated {len(waves)} profiles with {N} grid points on a period P={P:.3g}.")

## Spectral decay across modes

The first panel shows coefficient magnitude against mode number; the second uses a logarithmic magnitude axis to make the tail visible. The dashed line is a least-squares exponential-tail fit on the selected modes for the largest-amplitude wave. Change `fit_modes` to inspect another range; the fitted slope is descriptive, not a universal decay rate.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for target_amp, magnitudes in zip(target_amplitudes, coefficient_magnitudes):
    label = f"target amplitude = {target_amp:g}"
    axes[0].plot(modes[1:], magnitudes[1:], marker=".", label=label)
    axes[1].semilogy(modes[1:], magnitudes[1:], marker=".", label=label)

axes[0].set(title="Fourier coefficients by mode", xlabel="Mode number", ylabel=r"$|\\hat{u}_k|$")
axes[1].set(title="Spectral tail (log scale)", xlabel="Mode number", ylabel=r"$|\\hat{u}_k|$")
for ax in axes:
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()

fit_modes = (4, min(25, N // 2 + 1))
fit_indices = np.arange(*fit_modes)
fit_values = coefficient_magnitudes[-1, fit_indices]
valid = np.isfinite(fit_values) & (fit_values > 0)
if np.count_nonzero(valid) >= 2:
    slope, intercept = np.polyfit(fit_indices[valid], np.log(fit_values[valid]), 1)
    axes[1].semilogy(
        fit_indices[valid], np.exp(intercept + slope * fit_indices[valid]),
        "k--", label=f"tail fit: exp({slope:.3f} k)"
    )
    axes[1].legend()
    print(f"Fitted log-amplitude slope over modes {fit_modes[0]}-{fit_modes[1] - 1}: {slope:.4f}")
else:
    print("Not enough positive coefficients in the selected range to fit a spectral tail.")

plt.tight_layout()
plt.show()

## Modal magnitudes during KdV evolution

Evolve the largest generated profile and plot several nonzero Fourier-mode magnitudes over time. A steadily translating wave should keep these magnitudes nearly constant; changes indicate numerical error or a departure from a steadily translating profile.

In [ ]:
dt = 1e-3
T = 0.5
save_every = 10
solver = KdV(P=P, dt=dt, T=T, u0=profiles[-1], save_every=save_every)
times, solution = solver.splitStep()

solution_hat = np.fft.rfft(solution, axis=1) / N
modal_magnitudes = np.abs(solution_hat)
selected_modes = [1, 2, 3, 4, 6, 8]

fig, ax = plt.subplots(figsize=(9, 5))
for mode in selected_modes:
    ax.plot(times, modal_magnitudes[:, mode], label=f"mode {mode} (k={wavenumbers[mode]:.2f})")
ax.set(title="Fourier-mode magnitudes during KdV evolution", xlabel="Time", ylabel=r"$|\\hat{u}_k(t)|$")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

relative_change = (modal_magnitudes[-1, selected_modes] - modal_magnitudes[0, selected_modes]) / modal_magnitudes[0, selected_modes]
for mode, change in zip(selected_modes, relative_change):
    print(f"Mode {mode}: relative magnitude change over T={times[-1]:.3g}: {change:+.3e}")